In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
df = pd.read_csv('2stRoundFlightdata.csv')
X = df.drop(columns=["price"])
y = df["price"]
print(X.shape, y.shape)

(14697, 58) (14697,)


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=X["flight_class"],
)
print(X_train.shape, X_test.shape)

(11757, 58) (2940, 58)


In [4]:
def evaluate(y_true, y_pred, name="model"):
    # log-scale metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    # back to rupees
    true_rs, pred_rs = np.expm1(y_true), np.expm1(y_pred)
    mae_rs = mean_absolute_error(true_rs, pred_rs)
    mape = np.mean(np.abs(true_rs - pred_rs) / true_rs) * 100
    print(f"{name}: log RMSE={rmse:.4f} | log MAE={mae:.4f} | R2={r2:.4f} | "
          f"MAE=Rs {mae_rs:,.0f} | MAPE={mape:.1f}%")

# Baseline: mean log price per class, learned from the training set only
class_means = y_train.groupby(X_train["flight_class"]).mean()
baseline_pred = X_test["flight_class"].map(class_means)

evaluate(y_test, baseline_pred, "Baseline (class mean)")

Baseline (class mean): log RMSE=0.4703 | log MAE=0.3464 | R2=0.7767 | MAE=Rs 14,168 | MAPE=34.2%


In [ ]:
'''
Ridge Regression.'''

# first round - Ridge: log RMSE=0.2494 | log MAE=0.1756 | R2=0.9394 | MAE=Rs 6,552 | MAPE=17.3%

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0))
ridge.fit(X_train, y_train)
evaluate(y_test, ridge.predict(X_test), "Ridge")

Ridge: log RMSE=0.2504 | log MAE=0.1754 | R2=0.9367 | MAE=Rs 6,291 | MAPE=17.4%


In [6]:
"""
Random forest"""

# first round output - Random Forest: log RMSE=0.1495 | log MAE=0.0838 | R2=0.9782 | MAE=Rs 3,784 | MAPE=8.4%

from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
evaluate(y_test, rf.predict(X_test), "Random Forest")

Random Forest: log RMSE=0.1469 | log MAE=0.0825 | R2=0.9782 | MAE=Rs 3,584 | MAPE=8.1%


In [7]:
'''
Gradient Boosting
'''

# 1st round result - Gradient Boosting: log RMSE=0.1539 | log MAE=0.0970 | R2=0.9769 | MAE=Rs 4,166 | MAPE=9.7%

from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.1,
    random_state=42,
)

hgb.fit(X_train, y_train)
evaluate(y_test, hgb.predict(X_test), "Gradient Boosting")

Gradient Boosting: log RMSE=0.1504 | log MAE=0.0925 | R2=0.9772 | MAE=Rs 3,863 | MAPE=9.1%


In [8]:
'''
Tuned Gradient Boosting
'''

''' 1st round result -
Best settings: {'min_samples_leaf': 10, 'max_leaf_nodes': 63, 'max_iter': 800, 'learning_rate': 0.03, 'l2_regularization': 0.1}
Best CV RMSE: 0.1435
Tuned Gradient Boosting: log RMSE=0.1498 | log MAE=0.0919 | R2=0.9781 | MAE=Rs 3,958 | MAPE=9.2'''

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "learning_rate": [0.03, 0.05, 0.1, 0.15],
    "max_iter": [300, 500, 800],
    "max_leaf_nodes": [15, 31, 63],
    "min_samples_leaf": [10, 20, 40],
    "l2_regularization": [0.0, 0.1, 1.0],
}

search = RandomizedSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    random_state=42,
)

search.fit(X_train, y_train)

print("Best settings:", search.best_params_)
print(f"Best CV RMSE: {-search.best_score_:.4f}")

best_hgb = search.best_estimator_

evaluate(y_test, best_hgb.predict(X_test), "Tuned Gradient Boosting")

Best settings: {'min_samples_leaf': 10, 'max_leaf_nodes': 63, 'max_iter': 800, 'learning_rate': 0.03, 'l2_regularization': 0.1}
Best CV RMSE: 0.1397
Tuned Gradient Boosting: log RMSE=0.1495 | log MAE=0.0911 | R2=0.9774 | MAE=Rs 3,824 | MAPE=8.9%


In [ ]:
'''
Extra Trees
'''

# 1st round result - Extra Trees: Kaggle score = 0.96051
# Extra Trees: log RMSE=0.1417 | log MAE=0.0817 | R2=0.9804 | MAE=Rs 3,676 | MAPE=8.1%

from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=300,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)

et.fit(X_train, y_train)
evaluate(y_test, et.predict(X_test), "Extra Trees")

Extra Trees: log RMSE=0.1423 | log MAE=0.0797 | R2=0.9796 | MAE=Rs 3,454 | MAPE=7.8%


In [10]:
'''
LightGBM
'''

# 1st round result - LightGBM: log RMSE=0.1472 | log MAE=0.0886 | R2=0.9789 | MAE=Rs 3,944 | MAPE=8.9%

from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=10,
    reg_lambda=0.1,
    random_state=42,
    verbose=-1,
)

lgbm.fit(X_train, y_train)
evaluate(y_test, lgbm.predict(X_test), "LightGBM")

LightGBM: log RMSE=0.1452 | log MAE=0.0861 | R2=0.9787 | MAE=Rs 3,675 | MAPE=8.5%


In [11]:
'''
XGBoost
'''

# 1st round result - XGBoost: Kaggle score = 0.95978
# XGBoost: log RMSE=0.1371 | log MAE=0.0830 | R2=0.9817 | MAE=Rs 3,750 | MAPE=8.3%

from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=800,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
)

xgb.fit(X_train, y_train)
evaluate(y_test, xgb.predict(X_test), "XGBoost")

XGBoost: log RMSE=0.1394 | log MAE=0.0821 | R2=0.9804 | MAE=Rs 3,497 | MAPE=8.1%


In [12]:
'''
CatBoost
'''

# 1st round result - CatBoost: log RMSE=0.1441 | log MAE=0.0927 | R2=0.9798 | MAE=Rs 3,952 | MAPE=9.1%


from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=800,
    learning_rate=0.05,
    depth=8,
    random_state=42,
    verbose=0,
)

cat.fit(X_train, y_train)
evaluate(y_test, cat.predict(X_test), "CatBoost")

CatBoost: log RMSE=0.1453 | log MAE=0.0919 | R2=0.9787 | MAE=Rs 3,726 | MAPE=9.0%


In [13]:
'''
Cross-validation comparison
'''

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Ridge": ridge,
    "Random Forest": rf,
    "Extra Trees": et,
    "Gradient Boosting (tuned)": best_hgb,
    "LightGBM": lgbm,
    "XGBoost": xgb,
    "CatBoost": cat,
}

def cv_metrics(model, X, y, cv):
    rows = []

    for tr, va in cv.split(X):
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        pred = m.predict(X.iloc[va])
        true = y.iloc[va]

        true_rs = np.expm1(true)
        pred_rs = np.expm1(pred)

        rows.append({
            "log RMSE": np.sqrt(mean_squared_error(true, pred)),
            "log MAE": mean_absolute_error(true, pred),
            "R2": r2_score(true, pred),
            "MAE (Rs)": mean_absolute_error(true_rs, pred_rs),
            "MAPE (%)": np.mean(
                np.abs(true_rs - pred_rs) / true_rs
            ) * 100,
        })

    return pd.DataFrame(rows)


results = []

for name, model in models.items():
    f = cv_metrics(model, X_train, y_train, cv)

    mean = f.mean()
    std = f.std()

    print(
        f"{name}: "
        f"log RMSE={mean['log RMSE']:.4f} (+/- {std['log RMSE']:.4f}) | "
        f"log MAE={mean['log MAE']:.4f} | "
        f"R2={mean['R2']:.4f} | "
        f"MAE=Rs {mean['MAE (Rs)']:,.0f} | "
        f"MAPE={mean['MAPE (%)']:.1f}% (+/- {std['MAPE (%)']:.1f})"
    )

    results.append({
        "model": name,
        **mean.round(4).to_dict()
    })

pd.DataFrame(results).sort_values(
    "log RMSE"
).reset_index(drop=True)

'''

Ridge: log RMSE=0.2503 (+/- 0.0060) | log MAE=0.1767 | R2=0.9372 | MAE=Rs 6,379 | MAPE=17.4% (+/- 0.3)
Random Forest: log RMSE=0.1425 (+/- 0.0053) | log MAE=0.0834 | R2=0.9796 | MAE=Rs 3,514 | MAPE=8.3% (+/- 0.2)
Extra Trees: log RMSE=0.1417 (+/- 0.0066) | log MAE=0.0819 | R2=0.9799 | MAE=Rs 3,410 | MAPE=8.1% (+/- 0.3)
Gradient Boosting (tuned): log RMSE=0.1418 (+/- 0.0051) | log MAE=0.0872 | R2=0.9798 | MAE=Rs 3,623 | MAPE=8.7% (+/- 0.3)
LightGBM: log RMSE=0.1425 (+/- 0.0041) | log MAE=0.0873 | R2=0.9797 | MAE=Rs 3,637 | MAPE=8.7% (+/- 0.3)
XGBoost: log RMSE=0.1364 (+/- 0.0043) | log MAE=0.0829 | R2=0.9814 | MAE=Rs 3,444 | MAPE=8.2% (+/- 0.3)
CatBoost: log RMSE=0.1456 (+/- 0.0039) | log MAE=0.0941 | R2=0.9788 | MAE=Rs 3,735 | MAPE=9.3% (+/- 0.3)

'''

Ridge: log RMSE=0.2534 (+/- 0.0038) | log MAE=0.1779 | R2=0.9359 | MAE=Rs 6,585 | MAPE=17.6% (+/- 0.3)
Random Forest: log RMSE=0.1405 (+/- 0.0039) | log MAE=0.0821 | R2=0.9803 | MAE=Rs 3,482 | MAPE=8.2% (+/- 0.1)
Extra Trees: log RMSE=0.1399 (+/- 0.0050) | log MAE=0.0801 | R2=0.9804 | MAE=Rs 3,419 | MAPE=8.0% (+/- 0.3)
Gradient Boosting (tuned): log RMSE=0.1396 (+/- 0.0024) | log MAE=0.0861 | R2=0.9805 | MAE=Rs 3,611 | MAPE=8.6% (+/- 0.2)
LightGBM: log RMSE=0.1405 (+/- 0.0030) | log MAE=0.0864 | R2=0.9803 | MAE=Rs 3,609 | MAPE=8.6% (+/- 0.1)
XGBoost: log RMSE=0.1336 (+/- 0.0039) | log MAE=0.0815 | R2=0.9822 | MAE=Rs 3,385 | MAPE=8.1% (+/- 0.2)
CatBoost: log RMSE=0.1417 (+/- 0.0036) | log MAE=0.0916 | R2=0.9799 | MAE=Rs 3,695 | MAPE=9.1% (+/- 0.1)


,model,log RMSE,log MAE,R2,MAE (Rs),MAPE (%)
0,XGBoost,0.1336,0.0815,0.9822,3384.5423,8.0961
1,Gradient Boosting (tuned),0.1396,0.0861,0.9805,3611.3361,8.5637
2,Extra Trees,0.1399,0.0801,0.9804,3418.7378,7.9618
3,Random Forest,0.1405,0.0821,0.9803,3481.7582,8.1708
4,LightGBM,0.1405,0.0864,0.9803,3608.9375,8.5886
5,CatBoost,0.1417,0.0916,0.9799,3695.3959,9.0635
6,Ridge,0.2534,0.1779,0.9359,6584.9104,17.5735


XGBoost performed the best overall, achieving the lowest log RMSE (0.1336) and highest R² (0.9822). Extra Trees achieved the lowest MAPE (8.0%). The low cross-validation variation indicates that the models are relatively stable across different data splits.

Now compared to the first iteration this is slightly better, Now we use this data preprocessing and the above models and see their performance on the public test data on Kaggle.


## Result on Kaggle:
- XGBoost : 0.95798
- Random forest : 0.95157
- Gradient Boosting : 0.95496
- ExtraTree : 0.95905

In [ ]:
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import KFold, cross_val_predict

cv = KFold(n_splits=5, shuffle=True, random_state=42)
blend = VotingRegressor([("xgb", xgb), ("et", et)])

oof_pred = cross_val_predict(blend, X_train, y_train, cv=cv)
evaluate(y_train, oof_pred, "Blend (out-of-fold on train)")

# One row per flight: its features plus actual price, predicted price and error
errors = X_train.copy()
errors["actual_rs"] = np.expm1(y_train)
errors["pred_rs"] = np.expm1(oof_pred)
errors["error_pct"] = (errors["pred_rs"] - errors["actual_rs"]).abs() / errors["actual_rs"] * 100

print(errors["error_pct"].describe().round(1))

'''
Blend (out-of-fold on train): log RMSE=0.1352 | log MAE=0.0801 | R2=0.9817 | MAE=Rs 3,340 | MAPE=7.9%

'''

Blend (out-of-fold on train): log RMSE=0.1329 | log MAE=0.0785 | R2=0.9824 | MAE=Rs 3,319 | MAPE=7.8%
count    11757.0
mean         7.8
std         10.7
min          0.0
25%          1.6
50%          4.1
75%          9.7
max        166.6
Name: error_pct, dtype: float64


On Kaggle : the result of the above model is Score: 0.96077